# TP 2 / 10 — Preprocessing et feature engineering

**M2 Actuariat — Abidjan**
**Suite du TP 1 (EDA).**

---

## Objectifs
- **Recoder proprement** les variables qui posent problème (`VehAge`, `VehMaxSpeed`, `SocioCateg`, `Garage`).
- **Encoder** les variables catégorielles et binaires.
- **Séparer train / test** de manière **stratifiée**.
- **Standardiser** les variables numériques sans fuite (fit sur train uniquement).
- Sauvegarder les jeux préparés pour les TP 3 à 10.

> Règle d'or de tout pipeline ML : **toute transformation apprise sur les données (moyenne, écart-type, encodage, regroupement) doit être ajustée sur le train uniquement, puis appliquée au test**. Sinon on triche.

**Durée estimée : ~45 min.**

---


## 0. Imports et chargement du fichier produit en TP 1


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

df = pd.read_csv("df_raw_with_target.csv")
print("Dimensions :", df.shape)
df.head(3)


## 1. Recodage de `VehAge`

Rappel TP 1 : la colonne contient des chaînes `'0'`, `'1'`, ..., `'5'`, `'6-7'`, `'8-9'`, `'10+'`. Plusieurs choix raisonnables :

| Stratégie | Avantage | Inconvénient |
|---|---|---|
| (A) Numérique : `'10+' → 10`, `'8-9' → 8.5` | conserve l'ordre, permet régression linéaire | hypothèse d'**effet linéaire** sur l'âge |
| (B) Catégorielle one-hot | aucune hypothèse de linéarité | perte d'ordre, plus de paramètres |
| (C) Ordinal (rank) | conserve l'ordre, robuste | hypothèse d'écart **constant** entre rangs |

Nous adoptons **(A)** pour le TP — c'est l'approche la plus utilisée en tarification. Vous pourrez tester (B) en TP 5.


In [ ]:
def parse_vehage(x):
    s = str(x).strip()
    if s == "10+":
        return 10.0
    if "-" in s:
        a, b = s.split("-")
        return (float(a) + float(b)) / 2
    return float(s)

df["VehAge_num"] = df["VehAge"].apply(parse_vehage)
print(df[["VehAge", "VehAge_num"]].drop_duplicates().sort_values("VehAge_num"))


## 2. Recodage de `VehMaxSpeed`

Mêmes intervalles : `'1-130 km/h'`, `'130-140 km/h'`, ..., `'220+ km/h'`. On prend le **milieu de l'intervalle**.


In [ ]:
def parse_speed(x):
    s = str(x).replace(" km/h", "").strip()
    if s == "220+":
        return 220.0
    if s.startswith("1-"):
        return (1 + 130) / 2  # premier bin "1-130"
    a, b = s.split("-")
    return (float(a) + float(b)) / 2

df["VehMaxSpeed_num"] = df["VehMaxSpeed"].apply(parse_speed)
print(df[["VehMaxSpeed", "VehMaxSpeed_num"]].drop_duplicates().sort_values("VehMaxSpeed_num"))


### Question 1
- Le recodage de `VehMaxSpeed` par le milieu de l'intervalle suppose une **uniformité** à l'intérieur du bin. Donnez un cas où cette hypothèse pourrait être problématique.
- Pour `VehAge`, le recodage `'10+' → 10` sous-estime systématiquement l'âge des vieux véhicules. Comment pourriez-vous le corriger (proposez 2 idées) ?

*Votre réponse :*


## 3. Regroupement des `SocioCateg` rares

Une modalité avec **trop peu d'observations** n'est pas « crédible » au sens actuariel (lien direct avec le TP 6). Elle pose aussi des problèmes pratiques au GLM : coefficient instable, p-value non interprétable, voire séparation parfaite.

**Stratégie retenue** : on regroupe toutes les modalités dont l'effectif est inférieur à un seuil dans une catégorie `"CSP_autre"`.


In [ ]:
SEUIL = 100  # à débattre

counts = df["SocioCateg"].value_counts()
rares = counts[counts < SEUIL].index.tolist()
print(f"Nombre de modalités SocioCateg : {len(counts)}")
print(f"Modalités < {SEUIL} obs : {len(rares)} → regroupées en 'CSP_autre'")

df["SocioCateg_g"] = df["SocioCateg"].where(~df["SocioCateg"].isin(rares), "CSP_autre")
print("\nNouvelle distribution :")
print(df["SocioCateg_g"].value_counts())


In [ ]:
# Visualisation : taux de BAD par modalité regroupée
rate = df.groupby("SocioCateg_g")["target"].agg(["mean", "count"]).sort_values("mean", ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
rate["mean"].plot(kind="bar", color="steelblue", edgecolor="k", ax=ax)
ax.axhline(df["target"].mean(), color="red", linestyle="--", label="Moyenne globale")
ax.set_title("Taux de BAD par SocioCateg regroupée")
ax.set_ylabel("P(BAD)")
ax.legend()
plt.tight_layout()
plt.show()
print(rate)


### Question 2
- Le seuil de regroupement (100 ici) est un **hyperparamètre métier**. Qu'arriverait-il avec un seuil de 10 ? De 1000 ?
- Une alternative est de regrouper non pas par effectif mais par **profil de risque similaire** (clustering supervisé). Citez un avantage et un inconvénient de cette approche.

*Votre réponse :*


## 4. Traitement des valeurs manquantes de `Garage`

Rappel TP 1 : `Garage` a ~68% de NA, et le fait d'avoir une valeur manquante est **informatif** sur la cible. On crée donc une modalité `"Unknown"` plutôt que de jeter la variable.


In [ ]:
df["Garage"] = df["Garage"].fillna("Unknown")
print(df["Garage"].value_counts())

rate = df.groupby("Garage")["target"].agg(["mean", "count"])
print("\nTaux de BAD par modalité :")
print(rate)


### Question 3
- Le taux de BAD est-il vraiment différent entre `"Unknown"` et les autres modalités ? Cette différence est-elle assez forte pour justifier de garder la variable ?
- En pratique, comment décide-t-on qu'une variable « vaut la peine d'être gardée » ? (Pensez aux outils du cours de Data Science : tests, IC, validation croisée.)

*Votre réponse :*


## 5. Encodage des variables binaires

`Gender` (Male/Female) et `MariStat` (Other/Alone) sont binaires : on les recode directement en 0/1, ce qui évite le one-hot inutile.


In [ ]:
df["Gender_F"]   = (df["Gender"]   == "Female").astype(int)
df["MariAlone"]  = (df["MariStat"] == "Alone").astype(int)

print(df[["Gender", "Gender_F", "MariStat", "MariAlone"]].head())


## 6. Sélection des variables finales

On retire les colonnes brutes remplacées, et la cible texte `y`.


In [ ]:
num_cols = [
    "LicAge", "DrivAge", "BonusMalus", "RiskVar", "HasKmLimit",
    "VehAge_num", "VehMaxSpeed_num",
    "Gender_F", "MariAlone",
]
cat_cols = [
    "SocioCateg_g", "VehUsage", "VehBody", "VehPrice",
    "VehEngine", "VehEnergy", "VehClass", "Garage",
]
target_col = "target"

df_model = df[num_cols + cat_cols + [target_col]].copy()
print("Dimensions du jeu prêt à encoder :", df_model.shape)
df_model.head(3)


## 7. Séparation train / test stratifiée

On stratifie sur la cible pour préserver le taux de BAD dans les deux sous-ensembles — crucial vu le déséquilibre (~8.7%).


In [ ]:
X = df_model.drop(columns=[target_col])
y = df_model[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train : {X_train.shape}, taux BAD = {y_train.mean():.3%}")
print(f"Test  : {X_test.shape},  taux BAD = {y_test.mean():.3%}")


### Question 4
- Pourquoi est-il important que les taux de BAD soient quasiment identiques dans le train et le test ?
- Que se passerait-il si l'on prenait `test_size=0.05` au lieu de `0.3` ? Avantages / inconvénients ?
- Si vous deviez aussi créer un **jeu de validation** (en plus du test), comment le découperiez-vous ?

*Votre réponse :*


## 8. One-hot encoding **fitté sur le train**

Très important : on apprend les modalités sur le train, puis on aligne le test sur les mêmes colonnes. Toute modalité présente uniquement dans le test sera ignorée (silencieusement) — c'est la même règle que pour la standardisation.


In [ ]:
# Étape 1 : one-hot sur le TRAIN (drop_first=True pour éviter la colinéarité dans le GLM)
X_train_oh = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)

# Étape 2 : on applique exactement les mêmes colonnes au test
X_test_oh = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

print(f"Train one-hot : {X_train_oh.shape}")
print(f"Test  one-hot : {X_test_oh.shape}")
print("\nColonnes après encodage :")
print(X_train_oh.columns.tolist())


### Question 5
- Pourquoi utilise-t-on `drop_first=True` ici ? Quel piège évite-t-on ?
- Que se passe-t-il si une modalité de `VehBody` apparaît uniquement dans le test ? Pourquoi notre `reindex` la gère correctement ?

*Votre réponse :*


## 9. Standardisation des variables numériques (fit sur train uniquement)

On centre-réduit pour faciliter la convergence du GLM régularisé (TP 5) et rendre les coefficients comparables. **On calcule moyenne et écart-type sur le train**, puis on transforme les deux.


In [ ]:
mu  = X_train_oh[num_cols].mean()
sig = X_train_oh[num_cols].std(ddof=0)

X_train_oh[num_cols] = (X_train_oh[num_cols] - mu) / sig
X_test_oh[num_cols]  = (X_test_oh[num_cols]  - mu) / sig

print("Moyennes (train) après standardisation :")
print(X_train_oh[num_cols].mean().round(3))
print("\nÉcarts-types (train) après standardisation :")
print(X_train_oh[num_cols].std().round(3))
print("\nLes moyennes du TEST ne valent PAS 0 (c'est normal, on a appliqué la moyenne du train) :")
print(X_test_oh[num_cols].mean().round(3))


### Question 6
- Que se passerait-il, mathématiquement, si on standardisait **avant** le split (sur tout le dataset) ?
- Quel concept général de ML est-on en train d'illustrer ici ? (Indice : commence par « fuite ».)
- La standardisation est-elle **nécessaire** pour la régression logistique non régularisée ? Et pour LightGBM (TP 7) ?

*Votre réponse :*


## 10. Sauvegarde pour les TP suivants

Quatre fichiers : `X_train`, `X_test`, `y_train`, `y_test`. Tous les TP suivants commenceront par les charger.


In [ ]:
X_train_oh.to_csv("X_train.csv", index=False)
X_test_oh.to_csv("X_test.csv",   index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv",   index=False)

print("Fichiers sauvegardés :")
for f, s in [("X_train.csv", X_train_oh.shape),
             ("X_test.csv",  X_test_oh.shape),
             ("y_train.csv", y_train.shape),
             ("y_test.csv",  y_test.shape)]:
    print(f"  {f} : {s}")


## 11. Récapitulatif

| Étape | Choix retenu | Pourquoi |
|---|---|---|
| `VehAge` texte | numérique milieu de classe | conserve l'ordre, économe en paramètres |
| `VehMaxSpeed` texte | numérique milieu de classe | idem |
| `SocioCateg` rares | regroupement < 100 obs | stabilité statistique + crédibilité |
| `Garage` NA | modalité `"Unknown"` | l'absence est informative |
| `Gender`, `MariStat` | binaire 0/1 | économe en paramètres |
| Autres catégorielles | one-hot avec `drop_first` | évite la colinéarité parfaite |
| Numériques | centrer-réduire sur le train | comparabilité, convergence GLM régularisé |
| Split | 70/30 stratifié | préserve le déséquilibre |

### Question 7 (synthèse)
- Lister **trois sources potentielles de fuite** (data leakage) que nous avons explicitement évitées dans ce TP.
- Citez **deux décisions de preprocessing** que vous changeriez et expliquez l'impact attendu sur la performance.

*Votre réponse :*

---
**Prochain TP : TP 3 — GLM logistique (cours de tarification).**
On utilisera `X_train.csv` / `X_test.csv` pour ajuster notre premier modèle de tarification.
